# Phishing Site Detection - Data Preprocessing

This is the first notebook in my project. Here I load the raw dataset, clean the URLs, and turn them into numbers so the machine learning models can work with them.

## 1. Import Libraries

First I import everything I need for this notebook.

In [1]:
import os
import pickle

import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer

nltk.download('punkt', quiet=True)

print('Libraries imported successfully.')

Libraries imported successfully.


## 2. Load the Dataset

I load the CSV file from the parent folder. Then I check the first few rows and some basic info to make sure the data looks correct.

In [2]:
DATA_PATH = os.path.join('..', 'phishing_site_urls.csv')

df = pd.read_csv(DATA_PATH)

print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (549346, 2)


,URL,Label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,bad
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,bad
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,bad
3,mail.printakid.com/www.online.americanexpress....,bad
4,thewhiskeydregs.com/wp-content/themes/widescre...,bad


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 549346 entries, 0 to 549345
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   URL     549346 non-null  str  
 1   Label   549346 non-null  str  
dtypes: str(2)
memory usage: 8.4 MB


In [4]:
print('Label distribution:')
print(df['Label'].value_counts())

Label distribution:
Label
good    392924
bad     156422
Name: count, dtype: int64


## 3. Text Preprocessing

Here I clean each URL. First I use `RegexpTokenizer` to split the URL into separate words and numbers and remove things like slashes and dots. Then I use `SnowballStemmer` to reduce each word to its base form, for example "running" becomes "run". The cleaned result is saved in a new column called `processed_text`.

In [5]:
tokenizer = RegexpTokenizer(r'[A-Za-z0-9]+')
stemmer = SnowballStemmer('english')

def preprocess_url(url: str) -> str:
    """Tokenise and stem a single URL string.

    Args:
        url: Raw URL string.

    Returns:
        Space-separated string of stemmed alphanumeric tokens.
    """
    tokens = tokenizer.tokenize(url.lower())
    stemmed = [stemmer.stem(t) for t in tokens]
    return ' '.join(stemmed)


df['processed_text'] = df['URL'].apply(preprocess_url)

print('Preprocessing complete. Sample output:')
df[['URL', 'processed_text']].head()

Preprocessing complete. Sample output:


,URL,processed_text
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,nobel it 70ffb52d079109dca5664cce6f317373782 l...
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,www dghjdgf com paypal co uk cycgi bin webscrc...
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,serviciosbi com paypal cgi bin get into herf s...
3,mail.printakid.com/www.online.americanexpress....,mail printakid com www onlin americanexpress c...
4,thewhiskeydregs.com/wp-content/themes/widescre...,thewhiskeydreg com wp content theme widescreen...


## 4. Feature Extraction

I use `CountVectorizer` to turn the cleaned text into numbers. It counts how many times each word appears in each URL. I fit it on all the data here and save it so the models later can use the same word list.

In [6]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['processed_text'])

print(f'Feature matrix shape: {X.shape}')
print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')

Feature matrix shape: (549346, 468010)
Vocabulary size: 468010


## 5. Save the Files

I save two files here. The first is `processed_data.csv` which contains the URLs, labels and the cleaned text. The second is `vectorizer.pkl` which is the saved vectorizer object. All the model notebooks will load these two files so they do not need to redo the cleaning.

In [7]:
OUTPUT_DIR = os.path.dirname(os.path.abspath('data_preprocessing.ipynb'))

# Save processed DataFrame
csv_path = os.path.join(OUTPUT_DIR, 'processed_data.csv')
df.to_csv(csv_path, index=False)
print(f'Processed data saved to: {csv_path}')

# Save fitted vectorizer
vec_path = os.path.join(OUTPUT_DIR, 'vectorizer.pkl')
with open(vec_path, 'wb') as f:
    pickle.dump(vectorizer, f)
print(f'Vectorizer saved to:      {vec_path}')

Processed data saved to: c:\Users\RANON TRASH\Desktop\final-year-project\phishing_site_root\preprocessing-dataset\processed_data.csv
Vectorizer saved to:      c:\Users\RANON TRASH\Desktop\final-year-project\phishing_site_root\preprocessing-dataset\vectorizer.pkl


## Summary

In this notebook I loaded the raw dataset, cleaned all the URLs, converted them to features, and saved the results. The next step is to go into the model notebooks and train the classifiers using `processed_data.csv`.